# Direct-Sum vs Jaccpot Backend

This notebook compares the direct-sum backend and the current `jaccpot` adapter on a Hermite-4 solve.
It is intended as a first integration notebook; Hermite-6/8 are not yet available through `jaccpot`.

In [ ]:
from pathlib import Path
import sys

import jax
import jax.numpy as jnp

from nornax import AarsethController, JaccpotForceModel, solve_adaptive_to_time, total_energy
from nornax.forces import DirectSumGravity

jax.config.update("jax_enable_x64", True)

repo_root = Path.cwd().resolve().parent
for sibling in (repo_root / "yggdrax", repo_root / "jaccpot"):
    if sibling.exists() and str(sibling) not in sys.path:
        sys.path.insert(0, str(sibling))

from jaccpot import FastMultipoleMethod

In [ ]:
positions = jnp.asarray([[-1.0, 0.0, 0.0], [1.0, 0.0, 0.0]])
velocities = jnp.asarray([[0.0, 0.2, 0.0], [0.0, -0.2, 0.0]])
masses = jnp.asarray([1.0, 1.0])
controller = AarsethController(eta=0.03, min_dt=1.0e-4, max_dt=5.0e-2)

direct_result = solve_adaptive_to_time(
    positions,
    velocities,
    masses,
    DirectSumGravity(),
    t_final=0.2,
    order=4,
    controller=controller,
    atol=1.0e-6,
)

solver = FastMultipoleMethod(preset="fast", basis="solidfmm")
jaccpot_result = solve_adaptive_to_time(
    positions,
    velocities,
    masses,
    JaccpotForceModel(solver),
    t_final=0.2,
    order=4,
    controller=controller,
    atol=1.0e-6,
    args={"leaf_size": 8, "max_order": 2, "jerk_mode": "fast_approx"},
)

print("direct accepted steps:", direct_result.dt_history.shape[0])
print("jaccpot accepted steps:", jaccpot_result.dt_history.shape[0])
print("direct energy:", float(total_energy(direct_result.final_state)))
print("jaccpot energy:", float(total_energy(jaccpot_result.final_state)))
print("position difference:", float(jnp.linalg.norm(direct_result.final_state.positions - jaccpot_result.final_state.positions)))